In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

DATA_PATH = Path('../data/raw/data.csv')
OUT_PATH  = Path('../reports/age_distribution_agentic.png')

# ── Step 1: Load ─────────────────────────────────────────────────────────────
# Defence: try UTF-8 first; fall back to latin-1 to survive non-ASCII bytes.
# sep=None + engine='python' lets pandas sniff the delimiter automatically.
try:
    df = pd.read_csv(DATA_PATH, encoding='utf-8', sep=None, engine='python')
except UnicodeDecodeError:
    df = pd.read_csv(DATA_PATH, encoding='latin-1', sep=None, engine='python')

total_rows = len(df)
print(f'Total rows loaded : {total_rows:,}')

# ── Step 2: Inspect dtype ────────────────────────────────────────────────────
# Defence: coerce age to numeric so stray strings ('N/A', '') become NaN
# instead of silently surviving the filter or raising a TypeError.
assert 'age' in df.columns, "'age' column not found — check the file."
df['age'] = pd.to_numeric(df['age'], errors='coerce')
n_coerced = df['age'].isna().sum()
if n_coerced:
    print(f'  Warning: {n_coerced:,} non-numeric age value(s) coerced to NaN')

# ── Step 3: Filter ───────────────────────────────────────────────────────────
# Drop NaN first so the boolean comparison is safe, then apply age bounds.
df_clean = df.dropna(subset=['age'])
df_clean = df_clean[(df_clean['age'] >= 13) & (df_clean['age'] <= 80)]
kept_rows = len(df_clean)
print(f'Rows kept (13-80) : {kept_rows:,}  '
      f'(dropped {total_rows - kept_rows:,})')

# ── Step 4: Plot ─────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(df_clean['age'], bins=30, color='steelblue',
        edgecolor='white', linewidth=0.6)
ax.set_xlabel('Age (years)', fontsize=12)
ax.set_ylabel('Count', fontsize=12)
ax.set_title(
    f'MGKT Mini Explorer — Age Distribution  (n = {kept_rows:,})',
    fontsize=13, fontweight='bold'
)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()

# ── Step 5: Save ─────────────────────────────────────────────────────────────
# mkdir ensures the reports/ folder exists even in a fresh checkout.
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(OUT_PATH, dpi=150)
plt.show()
print(f'Figure saved to   : {OUT_PATH.resolve()}')